# 第一阶段：环境安装与数据集准备
VisDrone2019-DET 格式转换为 YOLO 格式

In [1]:
# 验证环境
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
print('环境正常，YOLOv11 加载成功')

环境正常，YOLOv11 加载成功


## 下载 VisDrone 数据集
从 Kaggle 下载后解压到 `data/VisDrone/`，目录结构如下：
```
data/VisDrone/
├── VisDrone2019-DET-train/
│   ├── images/
│   └── annotations/
├── VisDrone2019-DET-val/
└── VisDrone2019-DET-test-dev/
```

In [2]:
from pathlib import Path
from PIL import Image

def convert_split(ann_dir, img_dir, label_dir):
    Path(label_dir).mkdir(parents=True, exist_ok=True)
    converted = 0
    for ann_file in Path(ann_dir).glob('*.txt'):
        img_path = Path(img_dir) / ann_file.with_suffix('.jpg').name
        if not img_path.exists():
            continue
        w, h = Image.open(img_path).size
        lines = []
        for line in ann_file.read_text().strip().splitlines():
            p = line.strip().split(',')
            if len(p) < 6:
                continue
            x, y, bw, bh, _, cls = int(p[0]), int(p[1]), int(p[2]), int(p[3]), p[4], int(p[5])
            if cls in (0, 11):
                continue
            lines.append(f'{cls-1} {(x+bw/2)/w:.6f} {(y+bh/2)/h:.6f} {bw/w:.6f} {bh/h:.6f}')
        (Path(label_dir) / ann_file.name).write_text('\n'.join(lines))
        converted += 1
    print(f'{label_dir}: {converted} 张')

base = Path('data/VisDrone')
for split, folder in [('train','VisDrone2019-DET-train'), ('val','VisDrone2019-DET-val'), ('test','VisDrone2019-DET-test-dev')]:
    convert_split(
        base / folder / 'annotations',
        base / folder / 'images',
        base / 'labels' / split
    )

data\VisDrone\labels\train: 6471 张
data\VisDrone\labels\val: 548 张
data\VisDrone\labels\test: 1610 张


In [3]:
# 统计数据集信息
import os
for split in ['train', 'val', 'test']:
    img_dir = base / f'VisDrone2019-DET-{split}' / 'images'
    if split == 'test':
        img_dir = base / 'VisDrone2019-DET-test-dev' / 'images'
    count = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    print(f'{split}: {count} 张图片')

train: 6471 张图片
val: 548 张图片
test: 1610 张图片
